In [0]:
# Databricks Notebook: validate_dim_date
from pyspark.sql import functions as F

print("Starting dim_date integrity validation...")

# 1. Read dim_date
df_date = spark.table("workspace.silver.dim_date")

# 2. Count total records
total_rows = df_date.count()
assert total_rows >= 7000, f"CRITICAL ERROR: Incomplete dim_date. Rows found: {total_rows} (expected > 7000)."

# 3. Validate sentinel record presence (-1)
sentinel_count = df_date.filter(F.col("DateKey") == -1).count()
assert sentinel_count == 1, f"CRITICAL ERROR: Missing sentinel record DateKey = -1 in dim_date."

# 4. Check date range coverage: historical (min 2016 for order 4) and future (min 2030)
metrics = df_date.filter(F.col("DateKey") != -1).select(
    F.min("DateKey").alias("min_key"),
    F.max("DateKey").alias("max_key")
).collect()[0]

min_date = metrics["min_key"]
max_date = metrics["max_key"]

assert min_date <= 20160101, f"CRITICAL ERROR: Insufficient historical coverage. Minimum date: {min_date} (expected <= 20160101)."
assert max_date >= 20301231, f"CRITICAL ERROR: Insufficient future coverage. Maximum date: {max_date} (expected >= 20301231)."

print(f"dim_date validated successfully. Rows: {total_rows} | Range: {min_date} to {max_date} | Sentinel present.")